In [1]:
print(0)

0


In [2]:
#!/usr/bin/env python3
"""
Patient data statistics for:
  - TRAIN_OPEN_SOURCE_FULL_DATA  → patient / images
  - TRAIN_REAL_PATIENT_FULL_DATA → patient / complaint / visit / images

Reports:
  - Counts: patients, complaints (RPD only), visits, images
  - Images per visit distribution (1-6, 7+)
  - Resolution stats: min / median / max for W, H, megapixels
  - Small-image counts
"""

import os
import math
from pathlib import Path
from collections import defaultdict
from PIL import Image

OSD_ROOT = "/home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TRAIN_OPEN_SOURCE_FULL_DATA"
RPD_ROOT = "/home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TRAIN_REAL_PATIENT_FULL_DATA"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

SMALL_THRESHOLDS = [(64, 64), (128, 128), (256, 256)]


# ── helpers ───────────────────────────────────────────────────────────────────

def read_dims(path):
    try:
        with Image.open(path) as im:
            return im.size          # (width, height)
    except Exception:
        return None


def percentile(data, p):
    if not data:
        return float("nan")
    s = sorted(data)
    idx = (len(s) - 1) * p / 100
    lo, hi = int(idx), math.ceil(idx)
    if lo == hi:
        return s[lo]
    return s[lo] * (hi - idx) + s[hi] * (idx - lo)


def get_images(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if Path(f).suffix.lower() in IMAGE_EXTS
    ])


def fmt(v, decimals=0):
    if math.isnan(v):
        return "     N/A"
    if decimals == 0:
        return f"{v:>8.0f}"
    return f"{v:>8.{decimals}f}"


def print_resolution_block(widths, heights, mps, indent="  "):
    n = len(widths)
    print(f"{indent}{'':30s} {'Min':>8} {'Median':>8} {'Max':>8}")
    print(f"{indent}{'Width  (px)':<30} "
          f"{fmt(min(widths) if widths else float('nan'))} "
          f"{fmt(percentile(widths, 50))} "
          f"{fmt(max(widths) if widths else float('nan'))}")
    print(f"{indent}{'Height (px)':<30} "
          f"{fmt(min(heights) if heights else float('nan'))} "
          f"{fmt(percentile(heights, 50))} "
          f"{fmt(max(heights) if heights else float('nan'))}")
    print(f"{indent}{'Megapixels':<30} "
          f"{fmt(min(mps) if mps else float('nan'), 3)} "
          f"{fmt(percentile(mps, 50), 3)} "
          f"{fmt(max(mps) if mps else float('nan'), 3)}")
    print(f"{indent}Small images (out of {n}):")
    for (tw, th) in SMALL_THRESHOLDS:
        c = sum(1 for w, h in zip(widths, heights) if w < tw or h < th)
        pct = 100 * c / n if n else 0
        print(f"{indent}  < {tw}×{th:<3}px : {c:>6}  ({pct:.1f}%)")


def print_ipc_distribution(counts_list, label="images/visit"):
    dist = defaultdict(int)
    for c in counts_list:
        bucket = str(c) if c <= 6 else "7+"
        dist[bucket] += 1
    print(f"  {label} distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7+"]:
        if dist[b]:
            print(f"    {b:>3} image(s) : {dist[b]:>6} visits")


# ── open source ───────────────────────────────────────────────────────────────

def analyse_osd():
    print("\n" + "=" * 70)
    print("  OPEN SOURCE DATA")
    print(f"  {OSD_ROOT}")
    print("=" * 70)

    patients = sorted([
        p for p in os.listdir(OSD_ROOT)
        if os.path.isdir(os.path.join(OSD_ROOT, p))
    ])
    n_patients = len(patients)

    all_widths, all_heights, all_mps = [], [], []
    errors = 0
    images_per_patient = []

    for i, pid in enumerate(patients, 1):
        ppath  = os.path.join(OSD_ROOT, pid)
        imgs   = get_images(ppath)
        images_per_patient.append(len(imgs))

        for imgpath in imgs:
            dims = read_dims(imgpath)
            if dims is None:
                errors += 1
                continue
            w, h = dims
            all_widths.append(w)
            all_heights.append(h)
            all_mps.append(w * h / 1_000_000)

        if i % 500 == 0:
            print(f"    ... {i}/{n_patients} patients", flush=True)

    n_images = len(all_widths) + errors
    print(f"\n  Patients      : {n_patients}")
    print(f"  Total images  : {n_images}  (read errors: {errors})")
    print()
    print_resolution_block(all_widths, all_heights, all_mps)
    print()
    print_ipc_distribution(images_per_patient, label="images/patient")


# ── real patient ──────────────────────────────────────────────────────────────

def analyse_rpd():
    print("\n" + "=" * 70)
    print("  REAL PATIENT DATA")
    print(f"  {RPD_ROOT}")
    print("=" * 70)

    patients = sorted([
        p for p in os.listdir(RPD_ROOT)
        if os.path.isdir(os.path.join(RPD_ROOT, p))
    ])
    n_patients = len(patients)

    n_complaints  = 0
    n_visits      = 0
    complaints_per_patient  = []
    visits_per_complaint    = []
    images_per_visit        = []

    all_widths, all_heights, all_mps = [], [], []
    errors = 0

    for i, pid in enumerate(patients, 1):
        ppath = os.path.join(RPD_ROOT, pid)
        complaints = sorted([
            c for c in os.listdir(ppath)
            if os.path.isdir(os.path.join(ppath, c))
        ])
        complaints_per_patient.append(len(complaints))
        n_complaints += len(complaints)

        for cid in complaints:
            cpath  = os.path.join(ppath, cid)
            visits = sorted([
                v for v in os.listdir(cpath)
                if os.path.isdir(os.path.join(cpath, v))
            ])
            visits_per_complaint.append(len(visits))
            n_visits += len(visits)

            for vid in visits:
                vpath = os.path.join(cpath, vid)
                imgs  = get_images(vpath)
                images_per_visit.append(len(imgs))

                for imgpath in imgs:
                    dims = read_dims(imgpath)
                    if dims is None:
                        errors += 1
                        continue
                    w, h = dims
                    all_widths.append(w)
                    all_heights.append(h)
                    all_mps.append(w * h / 1_000_000)

        if i % 200 == 0:
            print(f"    ... {i}/{n_patients} patients", flush=True)

    n_images = len(all_widths) + errors

    print(f"\n  Patients            : {n_patients}")
    print(f"  Total complaints    : {n_complaints}  "
          f"(min={min(complaints_per_patient)}  "
          f"median={percentile(complaints_per_patient,50):.0f}  "
          f"max={max(complaints_per_patient)} per patient)")
    print(f"  Total visits        : {n_visits}  "
          f"(min={min(visits_per_complaint)}  "
          f"median={percentile(visits_per_complaint,50):.0f}  "
          f"max={max(visits_per_complaint)} per complaint)")
    print(f"  Total images        : {n_images}  (read errors: {errors})")
    print()
    print_resolution_block(all_widths, all_heights, all_mps)
    print()
    print_ipc_distribution(images_per_visit, label="images/visit")

    # complaints per patient distribution
    dist = defaultdict(int)
    for c in complaints_per_patient:
        bucket = str(c) if c <= 8 else "9+"
        dist[bucket] += 1
    print(f"\n  complaints/patient distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7", "8", "9+"]:
        if dist[b]:
            print(f"    {b:>3} complaint(s) : {dist[b]:>6} patients")

    # visits per complaint distribution
    dist2 = defaultdict(int)
    for v in visits_per_complaint:
        bucket = str(v) if v <= 8 else "9+"
        dist2[bucket] += 1
    print(f"\n  visits/complaint distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7", "8", "9+"]:
        if dist2[b]:
            print(f"    {b:>3} visit(s) : {dist2[b]:>6} complaints")


# ── main ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    analyse_osd()
    analyse_rpd()
    print("\n" + "=" * 70)
    print("  DONE")
    print("=" * 70 + "\n")


  OPEN SOURCE DATA
  /home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TRAIN_OPEN_SOURCE_FULL_DATA
    ... 500/2624 patients
    ... 1000/2624 patients
    ... 1500/2624 patients
    ... 2000/2624 patients
    ... 2500/2624 patients

  Patients      : 2624
  Total images  : 4110  (read errors: 0)

                                      Min   Median      Max
  Width  (px)                          99      739     3872
  Height (px)                          87      810     2592
  Megapixels                        0.011    0.590   10.036
  Small images (out of 4110):
    < 64×64 px :      0  (0.0%)
    < 128×128px :     16  (0.4%)
    < 256×256px :    156  (3.8%)

  images/patient distribution:
      1 image(s) :   1809 visits
      2 image(s) :    144 visits
      3 image(s) :    671 visits

  REAL PATIENT DATA
  /home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TRAIN_REAL_PATIENT_FULL_DATA
    ... 200/5963 patients
    ... 400/5963 patients
    ... 600/5963 patients
    ... 800/5963 patients
   

In [3]:
#!/usr/bin/env python3
"""
Patient data statistics for:
  - TRAIN_OPEN_SOURCE_FULL_DATA  → patient / images
  - TRAIN_REAL_PATIENT_FULL_DATA → patient / complaint / visit / images

Reports:
  - Counts: patients, complaints (RPD only), visits, images
  - Images per visit distribution (1-6, 7+)
  - Resolution stats: min / median / max for W, H, megapixels
  - Small-image counts
"""

import os
import math
from pathlib import Path
from collections import defaultdict
from PIL import Image

OSD_ROOT = "/home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TEST_OPEN_SOURCE_FULL_DATA"
RPD_ROOT = "/home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TEST_REAL_PATIENT_FULL_DATA"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

SMALL_THRESHOLDS = [(64, 64), (128, 128), (256, 256)]


# ── helpers ───────────────────────────────────────────────────────────────────

def read_dims(path):
    try:
        with Image.open(path) as im:
            return im.size          # (width, height)
    except Exception:
        return None


def percentile(data, p):
    if not data:
        return float("nan")
    s = sorted(data)
    idx = (len(s) - 1) * p / 100
    lo, hi = int(idx), math.ceil(idx)
    if lo == hi:
        return s[lo]
    return s[lo] * (hi - idx) + s[hi] * (idx - lo)


def get_images(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if Path(f).suffix.lower() in IMAGE_EXTS
    ])


def fmt(v, decimals=0):
    if math.isnan(v):
        return "     N/A"
    if decimals == 0:
        return f"{v:>8.0f}"
    return f"{v:>8.{decimals}f}"


def print_resolution_block(widths, heights, mps, indent="  "):
    n = len(widths)
    print(f"{indent}{'':30s} {'Min':>8} {'Median':>8} {'Max':>8}")
    print(f"{indent}{'Width  (px)':<30} "
          f"{fmt(min(widths) if widths else float('nan'))} "
          f"{fmt(percentile(widths, 50))} "
          f"{fmt(max(widths) if widths else float('nan'))}")
    print(f"{indent}{'Height (px)':<30} "
          f"{fmt(min(heights) if heights else float('nan'))} "
          f"{fmt(percentile(heights, 50))} "
          f"{fmt(max(heights) if heights else float('nan'))}")
    print(f"{indent}{'Megapixels':<30} "
          f"{fmt(min(mps) if mps else float('nan'), 3)} "
          f"{fmt(percentile(mps, 50), 3)} "
          f"{fmt(max(mps) if mps else float('nan'), 3)}")
    print(f"{indent}Small images (out of {n}):")
    for (tw, th) in SMALL_THRESHOLDS:
        c = sum(1 for w, h in zip(widths, heights) if w < tw or h < th)
        pct = 100 * c / n if n else 0
        print(f"{indent}  < {tw}×{th:<3}px : {c:>6}  ({pct:.1f}%)")


def print_ipc_distribution(counts_list, label="images/visit"):
    dist = defaultdict(int)
    for c in counts_list:
        bucket = str(c) if c <= 6 else "7+"
        dist[bucket] += 1
    print(f"  {label} distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7+"]:
        if dist[b]:
            print(f"    {b:>3} image(s) : {dist[b]:>6} visits")


# ── open source ───────────────────────────────────────────────────────────────

def analyse_osd():
    print("\n" + "=" * 70)
    print("  OPEN SOURCE DATA")
    print(f"  {OSD_ROOT}")
    print("=" * 70)

    patients = sorted([
        p for p in os.listdir(OSD_ROOT)
        if os.path.isdir(os.path.join(OSD_ROOT, p))
    ])
    n_patients = len(patients)

    all_widths, all_heights, all_mps = [], [], []
    errors = 0
    images_per_patient = []

    for i, pid in enumerate(patients, 1):
        ppath  = os.path.join(OSD_ROOT, pid)
        imgs   = get_images(ppath)
        images_per_patient.append(len(imgs))

        for imgpath in imgs:
            dims = read_dims(imgpath)
            if dims is None:
                errors += 1
                continue
            w, h = dims
            all_widths.append(w)
            all_heights.append(h)
            all_mps.append(w * h / 1_000_000)

        if i % 500 == 0:
            print(f"    ... {i}/{n_patients} patients", flush=True)

    n_images = len(all_widths) + errors
    print(f"\n  Patients      : {n_patients}")
    print(f"  Total images  : {n_images}  (read errors: {errors})")
    print()
    print_resolution_block(all_widths, all_heights, all_mps)
    print()
    print_ipc_distribution(images_per_patient, label="images/patient")


# ── real patient ──────────────────────────────────────────────────────────────

def analyse_rpd():
    print("\n" + "=" * 70)
    print("  REAL PATIENT DATA")
    print(f"  {RPD_ROOT}")
    print("=" * 70)

    patients = sorted([
        p for p in os.listdir(RPD_ROOT)
        if os.path.isdir(os.path.join(RPD_ROOT, p))
    ])
    n_patients = len(patients)

    n_complaints  = 0
    n_visits      = 0
    complaints_per_patient  = []
    visits_per_complaint    = []
    images_per_visit        = []

    all_widths, all_heights, all_mps = [], [], []
    errors = 0

    for i, pid in enumerate(patients, 1):
        ppath = os.path.join(RPD_ROOT, pid)
        complaints = sorted([
            c for c in os.listdir(ppath)
            if os.path.isdir(os.path.join(ppath, c))
        ])
        complaints_per_patient.append(len(complaints))
        n_complaints += len(complaints)

        for cid in complaints:
            cpath  = os.path.join(ppath, cid)
            visits = sorted([
                v for v in os.listdir(cpath)
                if os.path.isdir(os.path.join(cpath, v))
            ])
            visits_per_complaint.append(len(visits))
            n_visits += len(visits)

            for vid in visits:
                vpath = os.path.join(cpath, vid)
                imgs  = get_images(vpath)
                images_per_visit.append(len(imgs))

                for imgpath in imgs:
                    dims = read_dims(imgpath)
                    if dims is None:
                        errors += 1
                        continue
                    w, h = dims
                    all_widths.append(w)
                    all_heights.append(h)
                    all_mps.append(w * h / 1_000_000)

        if i % 200 == 0:
            print(f"    ... {i}/{n_patients} patients", flush=True)

    n_images = len(all_widths) + errors

    print(f"\n  Patients            : {n_patients}")
    print(f"  Total complaints    : {n_complaints}  "
          f"(min={min(complaints_per_patient)}  "
          f"median={percentile(complaints_per_patient,50):.0f}  "
          f"max={max(complaints_per_patient)} per patient)")
    print(f"  Total visits        : {n_visits}  "
          f"(min={min(visits_per_complaint)}  "
          f"median={percentile(visits_per_complaint,50):.0f}  "
          f"max={max(visits_per_complaint)} per complaint)")
    print(f"  Total images        : {n_images}  (read errors: {errors})")
    print()
    print_resolution_block(all_widths, all_heights, all_mps)
    print()
    print_ipc_distribution(images_per_visit, label="images/visit")

    # complaints per patient distribution
    dist = defaultdict(int)
    for c in complaints_per_patient:
        bucket = str(c) if c <= 8 else "9+"
        dist[bucket] += 1
    print(f"\n  complaints/patient distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7", "8", "9+"]:
        if dist[b]:
            print(f"    {b:>3} complaint(s) : {dist[b]:>6} patients")

    # visits per complaint distribution
    dist2 = defaultdict(int)
    for v in visits_per_complaint:
        bucket = str(v) if v <= 8 else "9+"
        dist2[bucket] += 1
    print(f"\n  visits/complaint distribution:")
    for b in ["1", "2", "3", "4", "5", "6", "7", "8", "9+"]:
        if dist2[b]:
            print(f"    {b:>3} visit(s) : {dist2[b]:>6} complaints")


# ── main ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    analyse_osd()
    analyse_rpd()
    print("\n" + "=" * 70)
    print("  DONE")
    print("=" * 70 + "\n")


  OPEN SOURCE DATA
  /home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TEST_OPEN_SOURCE_FULL_DATA

  Patients      : 250
  Total images  : 352  (read errors: 0)

                                      Min   Median      Max
  Width  (px)                         160      670     2560
  Height (px)                         160      656     2208
  Megapixels                        0.026    0.460    4.915
  Small images (out of 352):
    < 64×64 px :      0  (0.0%)
    < 128×128px :      0  (0.0%)
    < 256×256px :      9  (2.6%)

  images/patient distribution:
      1 image(s) :    192 visits
      2 image(s) :     14 visits
      3 image(s) :     44 visits

  REAL PATIENT DATA
  /home/pratyushprasad/ACL/DATA/TRAIN_TEST_DATA/TEST_REAL_PATIENT_FULL_DATA
    ... 200/1735 patients
    ... 400/1735 patients
    ... 600/1735 patients
    ... 800/1735 patients
    ... 1000/1735 patients
    ... 1200/1735 patients
    ... 1400/1735 patients
    ... 1600/1735 patients

  Patients            : 1735
  To